<a href="https://colab.research.google.com/github/wiuver/tipis/blob/main/itz9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Лабораторная работа №9. Классификация текстов: наивный байесовский классификатор

Порядок выполнения работы:

1. Сформировать выборку текстов, разбитых на группы спам/не спам для обучающей выборки (в соотношении 4/4) и для тестовой (2/2). Источник текстов.
2. Убрать стоп-слова (предлоги, союзы, частицы, междометия, числа и пр.).
3. Произвести лемматизацию слов с приведением к нижнему регистру.
4. Сформировать набор уникальных слов, рассчитываем количество вхождений элементов массива в тексты обоих классов.
5. По формуле (2) произвести расчёты условных вероятностей p(xi|Qk). Параметр сглаживания принять равным 1.
6. Рассчитать вероятности принадлежности текста из тестовой выборки к классам «Спам» и «Не спам» по формуле (3).
7. Провести анализ результатов и оформление отчёта о проделанной работе.

In [ ]:
!pip install natasha

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 114.7 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=a62fed073ace28c947051d0bed071ce1ef3304399a01b4a74f58b1283ea894b7
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
  Created wheel for intervaltree: filename=intervaltree-3.1.0-py2.py3-none-any.whl size=26098 sha256=9bc2aec669328d593e759d32033b0f1b4c6dd34d83d45060321cfd50ea8d857e
  Stored in directory: /root/.cache/pip/wheels/65/c3/c3/238bf93c243597857edd94ddb0577faa74a8e16e9585896e83
Successfully built docopt intervaltree


In [ ]:
import natasha as n
import numpy as np
import string

f=open(r'/content/it.txt')
spam1=f.readline()
spam2=f.readline()
nspam1=f.readline()
nspam2=f.readline()
test=f.readline()
f.close()

doc1=n.Doc(spam1)
doc2=n.Doc(spam2)
doc3=n.Doc(nspam1)
doc4=n.Doc(nspam2)
doc5=n.Doc(test)
print(doc5)
#Разбиваем текст на токены и сегменты
segmenter = n.Segmenter()
doc1.segment(segmenter)
doc1.tokens
#print(doc1.tokens)
segmenter = n.Segmenter()
doc2.segment(segmenter)
doc2.tokens
#print(doc2.tokens)
segmenter = n.Segmenter()
doc3.segment(segmenter)
doc3.tokens
#print(doc3.tokens)
segmenter = n.Segmenter()
doc4.segment(segmenter)
doc4.tokens
#print(doc4.tokens)
segmenter = n.Segmenter()
doc5.segment(segmenter)
doc5.tokens
#print(doc5.tokens)

#Морфологический анализ
emb = n.NewsEmbedding()
morph_tagger = n.NewsMorphTagger(emb)
doc1.tag_morph(morph_tagger)
doc1.sents[0].morph.print()

doc2.tag_morph(morph_tagger)
doc2.sents[0].morph.print()

doc3.tag_morph(morph_tagger)
doc3.sents[0].morph.print()

doc4.tag_morph(morph_tagger)
doc4.sents[0].morph.print()

doc5.tag_morph(morph_tagger)
doc5.sents[0].morph.print()

#Лемматизация
morph_vocab = n.MorphVocab()
for token in doc1.tokens:
    token.lemmatize(morph_vocab)
##for token in doc1.tokens:
##    print(token.text,' : ', token.lemma)

morph_vocab = n.MorphVocab()
for token in doc2.tokens:
    token.lemmatize(morph_vocab)
##for token in doc2.tokens:
##    print(token.text,' : ', token.lemma)

morph_vocab = n.MorphVocab()
for token in doc3.tokens:
    token.lemmatize(morph_vocab)
##for token in doc3.tokens:
##    print(token.text,' : ', token.lemma)

morph_vocab = n.MorphVocab()
for token in doc4.tokens:
    token.lemmatize(morph_vocab)
##for token in doc4.tokens:
##    print(token.text,' : ', token.lemma)

morph_vocab = n.MorphVocab()
for token in doc5.tokens:
    token.lemmatize(morph_vocab)
##for token in doc5.tokens:
##    print(token.text,' : ', token.lemma)


#Удаляем стоп-слова
spam1words = []
for token in doc1.tokens:
    if token.pos == 'PUNCT' \
    or token.pos == 'ADP' \
    or token.pos == 'CCONJ' \
    or token.pos == 'NUM' \
    or token.pos == 'SYM':
        pass
    else:
        spam1words.append(token.lemma)
##print('Words: ', spam1words, len(spam1words))
##print('Unique: ',set(spam1words),len(set(spam1words)))

spam2words = []
for token in doc2.tokens:
    if token.pos == 'PUNCT' \
    or token.pos == 'ADP' \
    or token.pos == 'CCONJ' \
    or token.pos == 'NUM' \
    or token.pos == 'SYM':
        pass
    else:
        spam2words.append(token.lemma)
##print('Words: ', spam2words, len(spam2words))
##print('Unique: ',set(spam2words),len(set(spam2words)))

spam3words = []
for token in doc3.tokens:
    if token.pos == 'PUNCT' \
    or token.pos == 'ADP' \
    or token.pos == 'CCONJ' \
    or token.pos == 'NUM' \
    or token.pos == 'SYM':
        pass
    else:
        spam3words.append(token.lemma)
##print('Words: ', spam3words, len(spam3words))
##print('Unique: ',set(spam3words),len(set(spam3words)))

spam4words = []
for token in doc4.tokens:
    if token.pos == 'PUNCT' \
    or token.pos == 'ADP' \
    or token.pos == 'CCONJ' \
    or token.pos == 'NUM' \
    or token.pos == 'SYM':
        pass
    else:
        spam4words.append(token.lemma)
##print('Words: ', spam4words, len(spam4words))
##print('Unique: ',set(spam4words),len(set(spam4words)))

spam5words = []
for token in doc5.tokens:
    if token.pos == 'PUNCT' \
    or token.pos == 'ADP' \
    or token.pos == 'CCONJ' \
    or token.pos == 'NUM' \
    or token.pos == 'SYM':
        pass
    else:
        spam5words.append(token.lemma)
##print('Words: ', spam5words, len(spam5words))
##print('Unique: ',set(spam5words),len(set(spam5words)))
spamm=spam1words+spam2words
nnspam=spam3words+spam4words
m=spam1words+spam2words+spam3words+spam4words+spam5words

dataset=np.unique(m)
#функция для подсчета вхождения уникальных слов массив слов
def isam(what,where):
    count = np.zeros(len(what))
    for i in range(len(what)):
        for j in range(len(where)):
            if what[i]==where[j]:
              count[i]=count[i]+1
    return count

datasetInSpam=isam(dataset,spam1words+spam2words)
datasetInHam=isam(dataset,spam3words+spam4words)

#кол-во слов без тестовых
#trainingSize = len(np.unique([spam1words, spam2words, spam3words, spam4words]))
trainingSize = 4
##print(datasetInSpam)
##print(datasetInHam)
alpha=1
#считаем p(xk|Qk) - в-сть вхождения xk в док. класса Qk
Px_spam=(alpha+datasetInSpam)/(alpha*trainingSize+len(spam1words+spam2words))
Px_ham=(alpha+datasetInHam)/(alpha*trainingSize+len(spam3words+spam4words))

##print(Px_spam)
##print(Px_ham)

PSpam=[]
PHam=[]

for i in range(len(dataset)):
    for j in range(len(spam5words)):
        if dataset[i] == spam5words[j]:

            PSpam.append(Px_spam[i])

for i in range(len(dataset)):
    for j in range(len(spam5words)):
        if dataset[i] == spam5words[j]:

            PHam.append(Px_ham[i])


#print('PSpam =',2/4*np.prod(PSpam))
#print('PHam =',2/4*np.prod(PHam))

print('Вероятность, что письмо - спам:',np.prod(PSpam))
print('Вероятность, что письмо - не спам:', np.prod(PHam))

Doc(text='Приветствуем, Андрей ! Представляем новинку BEEF ...)
        Здравствуйте VERB|Aspect=Imp|Mood=Imp|Number=Plur|Person=2|VerbForm=Fin|Voice=Act
                   ! PUNCT
        Приветствуем VERB|Aspect=Imp|Mood=Ind|Number=Plur|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
                   ! PUNCT
                   " PUNCT
        Здравствуйте VERB|Aspect=Imp|Mood=Imp|Number=Plur|Person=2|VerbForm=Fin|Voice=Act
                   ! PUNCT
                   " PUNCT
        Здравствуйте VERB|Aspect=Imp|Mood=Imp|Number=Plur|Person=2|VerbForm=Fin|Voice=Act
                   ! PUNCT
        Приветствуем VERB|Aspect=Imp|Mood=Ind|Number=Plur|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
                   , PUNCT
              Андрей PROPN|Animacy=Anim|Case=Nom|Gender=Masc|Number=Sing
                   ! PUNCT
Вероятность, что письмо - спам: 2.654950743515644e-25
Вероятность, что письмо - не спам: 1.3580216908966164e-27
